# TorGen v2: Analysis Notebook

Diagnostic plots comparing generated tornado tracks against ground truth.

**Runtime:** Colab CPU (no GPU required).  
**Data:** `samples.parquet` + `gt_tracks.parquet` (both generated by `run_evaluation`) on Google Drive.

In [ ]:
!pip install -q git+https://github.com/jcaramichaellehigh/TorGen.git cartopy
!pip show torgen | grep -E "^(Version|Location)"
!python -c "
import json, urllib.request, importlib.metadata
dist = importlib.metadata.distribution('torgen')
du = dist.read_text('direct_url.json')
if du:
    sha = json.loads(du).get('vcs_info', {}).get('commit_id', '')
    if sha:
        try:
            url = f'https://api.github.com/repos/jcaramichaellehigh/TorGen/commits/{sha}'
            resp = json.loads(urllib.request.urlopen(url).read())
            msg = resp['commit']['message'].split('\n')[0]
            print(f'{sha[:8]} {msg}')
        except Exception:
            print(f'{sha[:8]} (could not fetch title)')
    else:
        print('No commit hash found')
else:
    print('Installed from non-git source')
"

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyproj
import cartopy.io.shapereader as shpreader
from scipy import stats

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# --- Paths (edit these) ---
EVAL_DIR = "/content/drive/MyDrive/checkpoints/eval"
RAW_DIR = "/content/drive/MyDrive/raw"  # .pt.gz files (only needed if parquets missing)
SAMPLES_PATH = os.path.join(EVAL_DIR, "samples.parquet")
GT_PATH = os.path.join(EVAL_DIR, "gt_tracks.parquet")
ENV_PATH = os.path.join(EVAL_DIR, "environment.parquet")
PLOTS_DIR = os.path.join(EVAL_DIR, "plots")

# --- NARR projection constants ---
NARR_PROJ = (
    "+proj=lcc +lon_0=-107 +lat_0=50 +x_0=5632730 +y_0=4612610 "
    "+lat_1=50 +lat_2=50 +ellps=sphere"
)
EASTING_MIN, EASTING_MAX = 5_323_932.0, 8_213_139.0
NORTHING_MIN, NORTHING_MAX = 1_817_928.0, 4_707_135.0

# Weather channel indices
WX_CHANNELS = {"stp": 8, "scp": 9, "acpcp": 5}

os.makedirs(PLOTS_DIR, exist_ok=True)

def _savefig(fig, name, dpi=150):
    """Save figure to Drive and display inline."""
    fig.savefig(os.path.join(PLOTS_DIR, f"{name}.png"), dpi=dpi, bbox_inches="tight")
    print(f"Saved: {name}.png")

In [ ]:
# --- State boundary helper ---
_STATE_CACHE = None

def _draw_states(ax, color="gray", linewidth=0.5, alpha=0.6):
    """Draw US state boundaries in [0,1] normalized NARR space."""
    global _STATE_CACHE
    if _STATE_CACHE is None:
        proj = pyproj.Proj(NARR_PROJ)
        shpfile = shpreader.natural_earth(
            resolution="110m", category="cultural",
            name="admin_1_states_provinces_lakes",
        )
        reader = shpreader.Reader(shpfile)
        segments = []
        e_range = EASTING_MAX - EASTING_MIN
        n_range = NORTHING_MAX - NORTHING_MIN
        for record in reader.records():
            if record.attributes.get("admin") != "United States of America":
                continue
            geom = record.geometry
            if geom.geom_type == "Polygon":
                polys = [geom]
            elif geom.geom_type == "MultiPolygon":
                polys = list(geom.geoms)
            else:
                continue
            for poly in polys:
                lons, lats = np.array(poly.exterior.coords).T
                ex, ny = proj(lons, lats)
                nx = (np.array(ex) - EASTING_MIN) / e_range
                ny_norm = (np.array(ny) - NORTHING_MIN) / n_range
                segments.append((nx, ny_norm))
        _STATE_CACHE = segments

    for nx, ny in _STATE_CACHE:
        ax.plot(nx, ny, color=color, linewidth=linewidth, alpha=alpha)

In [ ]:
# --- Load generated tracks ---
gen_df = pd.read_parquet(SAMPLES_PATH)
print(f"Generated tracks: {len(gen_df):,} rows, "
      f"{gen_df['date'].nunique()} days, "
      f"{gen_df['realization_id'].max() + 1} realizations/day")

In [ ]:
# --- Build gt_tracks.parquet + environment.parquet if missing (one-time, slow) ---
need_gt = not os.path.exists(GT_PATH)
need_env = not os.path.exists(ENV_PATH)

if need_gt or need_env:
    import torch
    from torgen.data.dataset import _load_pt, coords_to_bearing_length, SPLIT_YEARS
    print(f"Building from .pt.gz files (one-time): gt={need_gt}, env={need_env}")
    years = SPLIT_YEARS["test"]
    all_files = sorted(
        f for f in os.listdir(RAW_DIR)
        if (f.endswith(".pt") or f.endswith(".pt.gz")) and int(f[:4]) in years
    )
    gt_rows, env_rows = [], []
    for i, fname in enumerate(all_files):
        sample = _load_pt(os.path.join(RAW_DIR, fname))
        date = sample["date"]
        if need_gt:
            tracks_bl = coords_to_bearing_length(sample["tracks"])
            for j in range(tracks_bl.shape[0]):
                t = tracks_bl[j]
                gt_rows.append({
                    "date": date,
                    "se": float(t[0]), "sn": float(t[1]),
                    "bearing": float(t[2]), "length": float(t[3]),
                    "width": float(t[4]), "ef": int(t[5]),
                })
        if need_env:
            wx = sample["wx"]
            env_row = {"date": date}
            for name, ch in WX_CHANNELS.items():
                if ch < wx.shape[0]:
                    env_row[f"p99_{name}"] = float(wx[ch].quantile(0.99))
            env_rows.append(env_row)
        if (i + 1) % 200 == 0:
            print(f"  {i + 1}/{len(all_files)} days processed...")
    if need_gt:
        pd.DataFrame(gt_rows).to_parquet(GT_PATH, index=False)
        print(f"Saved {GT_PATH} ({len(gt_rows)} tracks)")
    if need_env:
        pd.DataFrame(env_rows).to_parquet(ENV_PATH, index=False)
        print(f"Saved {ENV_PATH} ({len(env_rows)} days)")
else:
    print("Using cached gt_tracks.parquet + environment.parquet")

In [ ]:
# --- Load GT tracks ---
gt_df = pd.read_parquet(GT_PATH)
print(f"GT tracks: {len(gt_df):,} across {gt_df['date'].nunique()} days")

# Build day-level stats
day_df = gt_df.groupby("date").size().reset_index(name="gt_count")
# Add days with zero GT tracks (they won't appear in gt_df)
all_gen_dates = gen_df["date"].unique()
day_df = pd.DataFrame({"date": all_gen_dates}).merge(day_df, on="date", how="left")
day_df["gt_count"] = day_df["gt_count"].fillna(0).astype(int)
print(f"{len(day_df)} days ({(day_df['gt_count'] == 0).sum()} null)")

# Merge generated count stats per day
n_realizations = gen_df["realization_id"].max() + 1
gen_counts = gen_df.groupby(["date", "realization_id"]).size().reset_index(name="count")
all_combos = pd.MultiIndex.from_product(
    [day_df["date"], range(n_realizations)], names=["date", "realization_id"]
).to_frame(index=False)
gen_counts = all_combos.merge(gen_counts, on=["date", "realization_id"], how="left")
gen_counts["count"] = gen_counts["count"].fillna(0).astype(int)

gen_day_stats = gen_counts.groupby("date")["count"].agg(
    mean_gen_count="mean", std_gen_count="std"
).reset_index()
day_df = day_df.merge(gen_day_stats, on="date", how="left")
day_df["std_gen_count"] = day_df["std_gen_count"].fillna(0)

## Count Distribution Analysis

In [ ]:
# --- Count histograms: GT vs Generated (log scale) ---
gt_day_counts = day_df["gt_count"].values
gen_real_counts = gen_counts["count"].values

max_count = max(gt_day_counts.max(), gen_real_counts.max())
bins = np.arange(-0.5, max_count + 1.5, 1)

ks_stat, ks_p = stats.ks_2samp(gt_day_counts, gen_real_counts)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(gt_day_counts, bins=bins, density=True, alpha=0.6, color="C0",
        label=f"Ground Truth (N={len(gt_day_counts)} days)")
ax.hist(gen_real_counts, bins=bins, density=True, alpha=0.6, color="C1",
        label=f"Generated (N={len(gen_real_counts):,} realizations)")
ax.set_yscale("log")
ax.set_xlabel("Tornado count per day/realization")
ax.set_ylabel("Density")
ax.set_title("Daily Tornado Count Distribution")
ax.legend()
ax.annotate(f"KS = {ks_stat:.3f}, p = {ks_p:.3g}",
            xy=(0.97, 0.95), xycoords="axes fraction",
            ha="right", va="top", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))
fig.tight_layout()
_savefig(fig, "count_histogram")
plt.show()

In [ ]:
# --- Per-day count scatter with ±1σ (log-log) ---
fig, ax = plt.subplots(figsize=(6, 6))

# Offset by +1 so zero counts are visible on log scale
gt_plot = day_df["gt_count"].values + 1
gen_plot = day_df["mean_gen_count"].values + 1
gen_err = day_df["std_gen_count"].values

ax.errorbar(
    gt_plot, gen_plot, yerr=gen_err,
    fmt="o", markersize=3, alpha=0.5, elinewidth=0.5, capsize=0, color="C0",
)
lim = max(gt_plot.max(), (gen_plot + gen_err).max()) * 1.2
ax.plot([0.8, lim], [0.8, lim], "--", color="gray", linewidth=0.8, label="1:1")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(0.8, lim)
ax.set_ylim(0.8, lim)
ax.set_xlabel("GT tornado count + 1")
ax.set_ylabel("Mean generated count + 1")
ax.set_title("Per-Day Count: GT vs Generated (±1σ, log scale)")
ax.legend()
ax.set_aspect("equal")
fig.tight_layout()
_savefig(fig, "count_scatter")
plt.show()

## Spatial Patterns

In [ ]:
# --- GT vs Generated genesis density ---
nbins = 20
hist_range = [[0, 1], [0, 1]]

gt_h, xedges, yedges = np.histogram2d(
    gt_df["se"].values, gt_df["sn"].values, bins=nbins, range=hist_range
)
gen_h, _, _ = np.histogram2d(
    gen_df["se"].values, gen_df["sn"].values, bins=nbins, range=hist_range
)
# Normalize generated by number of realizations so scale is comparable
gen_h = gen_h / n_realizations

vmax = max(gt_h.max(), gen_h.max())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
for ax, h, title in [(ax1, gt_h, "Ground Truth"), (ax2, gen_h, "Generated (per realization)")]:
    im = ax.imshow(
        h.T, origin="lower", extent=[0, 1, 0, 1],
        vmin=0, vmax=vmax, cmap="YlOrRd", aspect="equal",
    )
    _draw_states(ax)
    ax.set_title(title)
    ax.set_xlabel("Easting (normalized)")
    ax.set_ylabel("Northing (normalized)")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

fig.colorbar(im, ax=[ax1, ax2], label="Track density", shrink=0.8)
fig.suptitle("Genesis Location Density", fontsize=14, y=1.02)
fig.tight_layout()
_savefig(fig, "genesis_density")
plt.show()

## Track Properties

In [ ]:
# --- Bearing distribution (rose plot) ---
# bearing_norm in [0,1] maps to [-pi, pi]: bearing = bearing_norm * 2*pi - pi
gt_bearing_rad = gt_df["bearing"].values * 2 * np.pi - np.pi
gen_bearing_rad = gen_df["bearing"].values * 2 * np.pi - np.pi

n_bins_rose = 36  # 10-degree bins
rose_bins = np.linspace(-np.pi, np.pi, n_bins_rose + 1)

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"projection": "polar"})
# GT
gt_counts_rose, _ = np.histogram(gt_bearing_rad, bins=rose_bins)
gt_density = gt_counts_rose / gt_counts_rose.sum()
# Generated
gen_counts_rose, _ = np.histogram(gen_bearing_rad, bins=rose_bins)
gen_density = gen_counts_rose / gen_counts_rose.sum()

bin_centers = 0.5 * (rose_bins[:-1] + rose_bins[1:])
width = rose_bins[1] - rose_bins[0]

ax.bar(bin_centers, gt_density, width=width, alpha=0.5, color="C0", label="GT")
ax.bar(bin_centers, gen_density, width=width, alpha=0.5, color="C1", label="Generated")
# 0 = North (up), bearing is compass-style: atan2(dx, dy)
ax.set_theta_zero_location("N")
ax.set_theta_direction(-1)  # clockwise
ax.set_title("Track Bearing Distribution", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
fig.tight_layout()
_savefig(fig, "bearing_rose")
plt.show()

In [ ]:
# --- Length distribution ---
ks_len, ks_len_p = stats.ks_2samp(gt_df["length"].values, gen_df["length"].values)

fig, ax = plt.subplots(figsize=(8, 4))
bins_len = np.linspace(0, max(gt_df["length"].max(), gen_df["length"].max()), 50)
ax.hist(gt_df["length"].values, bins=bins_len, density=True, alpha=0.6,
        color="C0", label=f"GT (N={len(gt_df):,})")
ax.hist(gen_df["length"].values, bins=bins_len, density=True, alpha=0.6,
        color="C1", label=f"Generated (N={len(gen_df):,})")
ax.set_xlabel("Track length (normalized)")
ax.set_ylabel("Density")
ax.set_title("Track Length Distribution")
ax.legend()
ax.annotate(f"KS = {ks_len:.3f}, p = {ks_len_p:.3g}",
            xy=(0.97, 0.95), xycoords="axes fraction",
            ha="right", va="top", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))
fig.tight_layout()
_savefig(fig, "length_distribution")
plt.show()

In [ ]:
# --- Width distribution ---
ks_wid, ks_wid_p = stats.ks_2samp(gt_df["width"].values, gen_df["width"].values)

fig, ax = plt.subplots(figsize=(8, 4))
bins_wid = np.linspace(0, max(gt_df["width"].max(), gen_df["width"].max()), 50)
ax.hist(gt_df["width"].values, bins=bins_wid, density=True, alpha=0.6,
        color="C0", label=f"GT (N={len(gt_df):,})")
ax.hist(gen_df["width"].values, bins=bins_wid, density=True, alpha=0.6,
        color="C1", label=f"Generated (N={len(gen_df):,})")
ax.set_xlabel("Track width (normalized)")
ax.set_ylabel("Density")
ax.set_title("Track Width Distribution")
ax.legend()
ax.annotate(f"KS = {ks_wid:.3f}, p = {ks_wid_p:.3g}",
            xy=(0.97, 0.95), xycoords="axes fraction",
            ha="right", va="top", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))
fig.tight_layout()
_savefig(fig, "width_distribution")
plt.show()

In [ ]:
# --- EF rating distribution ---
ef_classes = list(range(6))
gt_ef_counts = gt_df["ef"].value_counts().reindex(ef_classes, fill_value=0)
gen_ef_counts = gen_df["ef"].value_counts().reindex(ef_classes, fill_value=0)

gt_ef_frac = gt_ef_counts / gt_ef_counts.sum()
gen_ef_frac = gen_ef_counts / gen_ef_counts.sum()

# Chi-squared test
chi2, chi2_p = stats.chisquare(gen_ef_counts.values, f_exp=gt_ef_frac.values * gen_ef_counts.sum())

x = np.arange(len(ef_classes))
w = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
bars_gt = ax.bar(x - w / 2, gt_ef_frac.values, w, label="GT", color="C0", alpha=0.8)
bars_gen = ax.bar(x + w / 2, gen_ef_frac.values, w, label="Generated", color="C1", alpha=0.8)

# Label bars with percentages
for bars in [bars_gt, bars_gen]:
    for bar in bars:
        h = bar.get_height()
        if h > 0.005:
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005,
                    f"{h * 100:.1f}%", ha="center", va="bottom", fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels([f"EF{c}" for c in ef_classes])
ax.set_xlabel("EF Rating")
ax.set_ylabel("Fraction")
ax.set_title("EF Rating Distribution")
ax.legend()
ax.annotate(f"χ² = {chi2:.1f}, p = {chi2_p:.3g}",
            xy=(0.97, 0.95), xycoords="axes fraction",
            ha="right", va="top", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))
fig.tight_layout()
_savefig(fig, "ef_distribution")
plt.show()

## Exists Probability Diagnostics

In [ ]:
# --- Exists probability analysis ---
EXISTS_PATH = os.path.join(EVAL_DIR, "exists_probs.parquet")

if not os.path.exists(EXISTS_PATH):
    print("exists_probs.parquet not found — re-run evaluation with latest torgen")
else:
    exists_df = pd.read_parquet(EXISTS_PATH)
    probs = exists_df["exists_prob"].values
    n_total = len(probs)
    pct_above = (probs > 0.5).mean() * 100
    print(f"All slots: {n_total:,} ({exists_df['date'].nunique()} days x "
          f"{exists_df['realization_id'].max()+1} realizations x "
          f"{exists_df['slot'].nunique()} slots)")
    print(f"  Mean: {probs.mean():.4f}  Median: {np.median(probs):.4f}  >0.5: {pct_above:.2f}%")

    # 1. Histogram of ALL slot exists probs
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    ax.hist(probs, bins=100, range=(0, 1), density=True, color="C0", alpha=0.7)
    ax.axvline(0.5, color="red", linestyle="--", linewidth=1, label="threshold=0.5")
    ax.set_yscale("log")
    ax.set_xlabel("Exists probability")
    ax.set_ylabel("Density (log scale)")
    ax.set_title(f"All Slots ({n_total:,.0f} total, {pct_above:.1f}% > 0.5)")
    ax.legend()

    # 2. Mean exists prob per slot index
    ax = axes[1]
    slot_means = exists_df.groupby("slot")["exists_prob"].mean()
    ax.bar(slot_means.index, slot_means.values, color="C1", alpha=0.7)
    ax.axhline(0.5, color="red", linestyle="--", linewidth=1)
    ax.set_xlabel("Query slot index")
    ax.set_ylabel("Mean exists probability")
    ax.set_title("Mean Exists Prob by Slot")

    fig.tight_layout()
    _savefig(fig, "exists_all_slots")
    plt.show()

    # 3. Threshold sensitivity
    gt_mean_count = day_df["gt_count"].mean()
    thresholds = np.arange(0.05, 1.0, 0.01)
    mean_counts = []
    n_real = exists_df.groupby(["date", "realization_id"]).ngroups
    for th in thresholds:
        counts_at_th = (exists_df["exists_prob"] > th).groupby(
            [exists_df["date"], exists_df["realization_id"]]
        ).sum()
        mean_counts.append(counts_at_th.mean())

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(thresholds, mean_counts, color="C0", linewidth=2)
    ax.axhline(gt_mean_count, color="gray", linestyle="--", linewidth=1,
               label=f"GT mean count = {gt_mean_count:.1f}")
    ax.axvline(0.5, color="red", linestyle="--", linewidth=1, alpha=0.5, label="current threshold")
    ax.set_xlabel("Exists threshold")
    ax.set_ylabel("Mean predicted count per realization")
    ax.set_title("Threshold Sensitivity: Count vs Threshold")
    ax.legend()
    ax.set_xlim(0, 1)
    fig.tight_layout()
    _savefig(fig, "threshold_sensitivity")
    plt.show()